#Tutorial: Clasificación de Imágenes mediante Función Discriminante Sintética (SDF)

Este repositorio contiene la implementación computacional de un arreglo de correladores ópticos de arquitectura 4f para la clasificación múltiple del dataset MNIST. A diferencia de las Redes Neuronales Convolucionales (CNN), este método no utiliza aprendizaje estadístico tradicional, sino que sintetiza un filtro de correlación espacial basado en el método SDF (Synthetic Discriminant Function).

##Clasificación Multiclase

### 1. Importación de Librerías y Preprocesamiento Físico Para el Dataset MNITS

En la simulación de sistemas ópticos, las imágenes no pueden ser procesadas tal cual se descargan. El comando `np.pad` se utiliza para expandir la imagen original de $28 \times 28$ a un tamaño físico simulado mayor ($256 \times 256$) rellenando con ceros. Esto es fundamental para evitar el efecto de convolución circular (aliasing) al aplicar Transformadas de Fourier.

Además, la función `zero_mean_normalize` cumple un doble propósito físico:

1.   Al restar la media, eliminamos el "componente DC" (frecuencia cero) que en un sistema óptico saturaría el centro del plano de correlación con luz de fondo.
2.   La normalización L2 iguala la "energía" de todas las imágenes para que el correlador no favorezca imágenes simplemente por ser más brillantes.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import numpy as np
import tensorflow as tf

print("Descargando el dataset MNIST desde TensorFlow/Keras (Alternativa a OpenML)...")
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

images = np.concatenate((train_images, test_images), axis=0)
labels = np.concatenate((train_labels, test_labels), axis=0).astype(int)

print(f"Éxito: {images.shape[0]} imágenes cargadas de tamaño 28x28.")

def pad_image(image, final_size=256):
    """Incrusta una imagen en el centro de un arreglo rellenado con ceros."""
    h, w = image.shape
    pad_h = (final_size - h) // 2
    pad_w = (final_size - w) // 2
    return np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant', constant_values=0)

def zero_mean_normalize(image):
    """Aplica media cero y normaliza la energía (L2) de la imagen para el correlador SDF."""
    img_zm = image - np.mean(image)
    energy = np.sqrt(np.sum(img_zm**2))
    if energy > 0:
        return img_zm / energy
    return img_zm

Descargando el dataset MNIST desde TensorFlow/Keras (Alternativa a OpenML)...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Éxito: 70000 imágenes cargadas de tamaño 28x28.


###2. Estructuración del Dataset

Para evaluar de manera justa el rendimiento del correlador, garantizamos un conjunto de prueba perfectamente balanceado. En este bloque, extraemos exactamente el mismo número de imágenes de prueba (`images_per_class_test`) para cada dígito. El uso de `np.random.shuffle` asegura que las tuplas seleccionadas para entrenamiento y prueba sean aleatorias y representativas.

In [2]:
images_per_class_test = 100
total_classes = 10

indices_prueba = []
indices_entrenamiento = []

print(f"Iniciando separación estructurada de datos...")

for i in range(total_classes):
    idx_clase = np.where(labels == i)[0]
    np.random.shuffle(idx_clase)

    seleccion_prueba = idx_clase[:images_per_class_test]
    seleccion_entrenamiento = idx_clase[images_per_class_test:]

    indices_prueba.extend(seleccion_prueba)
    indices_entrenamiento.extend(seleccion_entrenamiento)

indices_prueba = np.array(indices_prueba)
indices_entrenamiento = np.array(indices_entrenamiento)

clases_unicas, conteos = np.unique(labels[indices_prueba], return_counts=True)
expected_total = images_per_class_test * total_classes

if len(indices_prueba) == expected_total and np.all(conteos == images_per_class_test):
    print(f"\n¡Verificación exitosa! Hay exactamente {images_per_class_test} imágenes de prueba por clase.")
    print(f"Total de imágenes para prueba: {len(indices_prueba)}")
    print(f"Total de imágenes disponibles para entrenamiento: {len(indices_entrenamiento)}")

    np.save(f'indices_muestra_{expected_total}.npy', indices_prueba)
    np.save('indices_entrenamiento_disponibles.npy', indices_entrenamiento)
else:
    print("\nERROR: La distribución de prueba no es uniforme. Revisa los arreglos.")

Iniciando separación estructurada de datos...

¡Verificación exitosa! Hay exactamente 100 imágenes de prueba por clase.
Total de imágenes para prueba: 1000
Total de imágenes disponibles para entrenamiento: 69000


###3. Correlation SDF

El filtro de correlación SDF se diseña para penalizar explícitamente los lóbulos laterales (sidelobes) mediante la inclusión de imágenes desplazadas.


*   **Desplazamientos espaciales (`np.roll`)**: Simulamos las traslaciones de la imagen en $d_s$ píxeles.
*   **Vector de control (`u_list`)**: Impone la condición teórica. Asignamos `1.0` solo a la imagen verdadera sin desplazar ($dx=0, dy=0$). A las imágenes desplazadas y a todas las imágenes falsas se les exige una salida de `0.0 `en el centro del plano de correlación.
*   **SVD y Regularización de Tikhonov**: Resolver el sistema de ecuaciones lineal matricial
$$V a = u$$computacionalmente es inestable debido al enorme tamaño de las imágenes vectorizadas y a la colinealidad de los datos. Para resolverlo, utilizamos TruncatedSVD para proyectar el problema en un subespacio ortogonal de menor dimensión. Luego, añadimos una pequeña constante (`lambda_reg * np.eye`) en la diagonal de la matriz de covarianza antes de calcular su pseudoinversa (`np.linalg.pinv`). Esto estabiliza la matriz y garantiza la convergencia computacional de los pesos $a$.


In [3]:
from sklearn.decomposition import TruncatedSVD
import numpy as np

final_size = 256
d = final_size * final_size
clases_todas = list(range(10))

idx_train_por_clase = {i: [] for i in clases_todas}
for idx in indices_entrenamiento:
    clase = int(labels[idx])
    idx_train_por_clase[clase].append(idx)

num_variaciones = 36        # Imágenes verdaderas
num_falsos_por_clase = 2    # 2 por cada una de las 9 clases = 18 falsas en total
d_s = 8
desplazamientos = [(0, 0), (d_s, 0), (-d_s, 0), (0, d_s), (0, -d_s)]

# Componentes principales a retener
n_components = 150

print(f"Iniciando síntesis de 10 Filtros Multiclase SDF...")
reference_images_sdf = {}

for clase_objetivo in clases_todas:
    print(f" -> Entrenando filtro para el dígito {clase_objetivo}...")
    X_T = []
    u_list = []

    # 1. OBJETIVOS VERDADEROS
    for idx in idx_train_por_clase[clase_objetivo][:num_variaciones]:
        img_norm = zero_mean_normalize(pad_image(images[idx], final_size))

        for dy, dx in desplazamientos:
            img_shifted = np.roll(img_norm, shift=(dy, dx), axis=(0, 1))
            X_T.append(img_shifted.flatten())
            u_list.append(1.0 if dy == 0 and dx == 0 else 0.0)

    # 2. OBJETIVOS FALSOS
    for clase_falsa in clases_todas:
        if clase_falsa == clase_objetivo:
            continue

        for idx in idx_train_por_clase[clase_falsa][:num_falsos_por_clase]:
            img_norm = zero_mean_normalize(pad_image(images[idx], final_size))

            for dy, dx in desplazamientos:
                img_shifted = np.roll(img_norm, shift=(dy, dx), axis=(0, 1))
                X_T.append(img_shifted.flatten())
                u_list.append(0.0)

    X_T_np = np.array(X_T)
    u = np.array(u_list)

    # 3. REDUCCIÓN SVD
    total_images_in_filter = len(X_T)
    current_components = min(n_components, total_images_in_filter - 1)

    svd = TruncatedSVD(n_components=current_components, random_state=42)
    X_reduced = svd.fit_transform(X_T_np)
    eigen_images = svd.components_

    # 4. SOLUCIÓN DEL SISTEMA EN EL SUBESPACIO REDUCIDO
    R_reduced = X_reduced.T @ X_reduced
    lambda_reg = 1e-4 * np.trace(R_reduced)
    R_reg = R_reduced + lambda_reg * np.eye(current_components)
    R_inv = np.linalg.pinv(R_reg)

    a_reduced = R_inv @ (X_reduced.T @ u)
    h_1d = a_reduced @ eigen_images

    # 5. NORMALIZACIÓN L2
    h_2d = h_1d.reshape((final_size, final_size))
    norma = np.linalg.norm(h_2d)
    if norma > 0:
        h_2d = (h_2d / norma) * 100

    reference_images_sdf[clase_objetivo] = h_2d

print("\n¡Los 10 filtros SDF Multiclase han sido generados con éxito!")

Iniciando síntesis de 10 Filtros Multiclase SDF...
 -> Entrenando filtro para el dígito 0...
 -> Entrenando filtro para el dígito 1...
 -> Entrenando filtro para el dígito 2...
 -> Entrenando filtro para el dígito 3...
 -> Entrenando filtro para el dígito 4...
 -> Entrenando filtro para el dígito 5...
 -> Entrenando filtro para el dígito 6...
 -> Entrenando filtro para el dígito 7...
 -> Entrenando filtro para el dígito 8...
 -> Entrenando filtro para el dígito 9...

¡Los 10 filtros SDF Multiclase han sido generados con éxito!


###4. Simulación del Correlador Óptico 4f y Clasificación

Este bloque modela matemáticamente la arquitectura física de un correlador de dos lentes. El teorema de Wiener-Khinchin nos permite calcular la correlación cruzada espacial en el dominio de la frecuencia.

Notemos el uso de `np.conj(F_filter)`: en un arreglo 4f real, la lente transformadora no multiplica las funciones directamente, sino que la luz incidente interactúa con la transmitancia del filtro adaptado. Tomar el conjugado complejo en el espacio de Fourier equivale a invertir espacialmente la imagen de referencia, lo que transforma matemáticamente la operación de convolución en correlación. Finalmente, `fftshift` recentra el resultado óptico para leer las intensidades (`np.abs(...)2`), y se emula la lógica multiclase asignando la predicción al filtro que haya generado el pico lumínico de mayor intensidad (`np.argmax`).

In [4]:
from numpy.fft import fft2, ifft2, fftshift

def optical_correlator(image, filter_h):
    F_img = fft2(image)
    F_filter = fft2(filter_h)
    correlation = fftshift(np.abs(ifft2(F_img * np.conj(F_filter)))**2)
    return correlation

y_true = []
y_pred = []

print(f"\nEvaluando {len(indices_prueba)} imágenes de prueba contra los 10 correladores...")
start_time = time.time()

for count, idx in enumerate(indices_prueba):
    if count % 200 == 0 and count > 0:
        print(f"    Progreso: {count}/{len(indices_prueba)} evaluadas...")

    test_img = pad_image(images[idx], final_size)
    test_img_norm = zero_mean_normalize(test_img)
    true_label = int(labels[idx])

    peak_intensities = np.zeros(10)

    # Correlación cruzada contra los 10 filtros
    for i in clases_todas:
        peak_intensities[i] = np.max(optical_correlator(test_img_norm, reference_images_sdf[i]))

    # El filtro con la máxima respuesta gana
    predicted_digit = np.argmax(peak_intensities)

    y_true.append(true_label)
    y_pred.append(predicted_digit)

y_true_np = np.array(y_true)
y_pred_np = np.array(y_pred)

elapsed_time = time.time() - start_time
print(f"\nTIEMPO DE PROCESAMIENTO: {elapsed_time:.2f} segundos")

print("\n" + "="*45)
print("   MÉTRICAS DE CLASIFICACIÓN MULTICLASE SDF")
print("="*45)
print(f"Precisión global (Accuracy): {accuracy_score(y_true_np, y_pred_np):.4f}\n")
print("Matriz de Confusión:\n", confusion_matrix(y_true_np, y_pred_np))
print("\nReporte detallado:")
print(classification_report(y_true_np, y_pred_np))


Evaluando 1000 imágenes de prueba contra los 10 correladores...
    Progreso: 200/1000 evaluadas...
    Progreso: 400/1000 evaluadas...
    Progreso: 600/1000 evaluadas...
    Progreso: 800/1000 evaluadas...

TIEMPO DE PROCESAMIENTO: 71.83 segundos

   MÉTRICAS DE CLASIFICACIÓN MULTICLASE SDF
Precisión global (Accuracy): 0.6240

Matriz de Confusión:
 [[66  1  4  3  2  3  6 12  3  0]
 [ 0 66  3  1  0  2  0 12 11  5]
 [ 5  0 59  8  6  0  9  9  2  2]
 [ 1  0  1 68  0  2  1  7  2 18]
 [ 0  0  2  0 82  0  7  2  6  1]
 [ 7  2  4 22  6 30  3  1  5 20]
 [ 2  1  1  0 16  5 69  2  4  0]
 [ 1  3  7  0  1  2  1 68 11  6]
 [ 2  3  1  3  4  3  8  3 55 18]
 [ 0  2  2  3 11  2  1  7 11 61]]

Reporte detallado:
              precision    recall  f1-score   support

           0       0.79      0.66      0.72       100
           1       0.85      0.66      0.74       100
           2       0.70      0.59      0.64       100
           3       0.63      0.68      0.65       100
           4       0.64 

##Clasificación Binaria

En esta sección, evaluamos el límite de operación del filtro de Función Discriminante Sintética (SDF) aislando el problema a una clasificación binaria (uno contra uno). El objetivo es enfrentar todos los pares posibles de dígitos del dataset MNIST (45 combinaciones en total) utilizando un conjunto de entrenamiento fuertemente restringido (10 verdaderos y 5 falsos por filtro).

### 1. Codigo general o agrupado

El arreglo desplazamientos contiene las 5 posiciones exigidas teóricamente: el centro exacto $(0,0)$ y las variaciones en las cuatro direcciones cardinales. Posteriormente, extraemos aleatoriamente las muestras para garantizar un entorno de prueba balanceado y preparamos una matriz de 10x10 para almacenar el accuracy de las 45 comparaciones.

Para cada par de dígitos (por ejemplo, 0 vs 1), se construyen dos filtros: uno donde el '0' es el objetivo verdadero y el '1' el falso, y viceversa

In [8]:
import numpy as np
import time
import pandas as pd
from numpy.fft import fft2, ifft2, fftshift
from sklearn.metrics import accuracy_score

final_size = 256
d = final_size * final_size
num_variaciones = 10
num_falsos = 5
d_s = 8
desplazamientos = [(0, 0), (d_s, 0), (-d_s, 0), (0, d_s), (0, -d_s)]
total_images_in_filter = (num_variaciones + num_falsos) * len(desplazamientos)
images_per_class_test = 100

def optical_correlator(image, filter_h):
    """Simulador óptico optimizado"""
    F_img = fft2(image)
    F_filter = fft2(filter_h)
    return fftshift(np.abs(ifft2(F_img * np.conj(F_filter)))**2)


print("Iniciando separación estructurada de datos (Clases 0 al 9)...")
np.random.seed(42)

idx_train_all = {i: [] for i in range(10)}
idx_test_all = {i: [] for i in range(10)}

for clase in range(10):
    idx_clase = np.where(labels == clase)[0]
    np.random.shuffle(idx_clase)

    idx_test_all[clase] = idx_clase[:images_per_class_test]
    idx_train_all[clase] = idx_clase[images_per_class_test:]

accuracy_matrix = np.full((10, 10), 'X', dtype=object)
for i in range(10):
    accuracy_matrix[i, i] = 1.0

# ==========================================
# 3. BUCLE PRINCIPAL (45 COMBINACIONES)
# ==========================================
start_time_total = time.time()
print("\nIniciando síntesis iterativa y evaluación de matriz SDF...")

# Iteramos sobre todas las combinaciones únicas
for clase_A in range(10):
    for clase_B in range(clase_A + 1, 10):
        print(f"\n---> Evaluando par: {clase_A} vs {clase_B}")

        # --- SÍNTESIS DE FILTROS ---
        reference_images_sdf = {}
        for clase_objetivo in [clase_A, clase_B]:
            clase_falsa = clase_B if clase_objetivo == clase_A else clase_A

            X = np.zeros((d, total_images_in_filter))
            u_list = []
            col = 0

            # Objetivos Verdaderos
            for idx in idx_train_all[clase_objetivo][:num_variaciones]:
                img_padded = pad_image(images[idx], final_size)
                img_norm = zero_mean_normalize(img_padded)

                for dy, dx in desplazamientos:
                    img_shifted = np.roll(img_norm, shift=(dy, dx), axis=(0, 1))
                    X[:, col] = img_shifted.flatten()
                    u_list.append(1.0 if dy == 0 and dx == 0 else 0.0)
                    col += 1

            # Objetivos Falsos
            for idx in idx_train_all[clase_falsa][:num_falsos]:
                img_padded = pad_image(images[idx], final_size)
                img_norm = zero_mean_normalize(img_padded)

                for dy, dx in desplazamientos:
                    img_shifted = np.roll(img_norm, shift=(dy, dx), axis=(0, 1))
                    X[:, col] = img_shifted.flatten()
                    u_list.append(0.0)
                    col += 1

            # Solución del sistema
            u = np.array(u_list)
            R = X.T @ X
            lambda_reg = 1e-4 * np.trace(R)
            R_inv = np.linalg.pinv(R + lambda_reg * np.eye(total_images_in_filter))
            a = R_inv @ u
            h_1d = X @ a
            reference_images_sdf[clase_objetivo] = h_1d.reshape((final_size, final_size))

        # --- CLASIFICACIÓN ÓPTICA ---
        indices_prueba_par = np.concatenate([idx_test_all[clase_A], idx_test_all[clase_B]])
        y_true = []
        y_pred = []

        for idx in indices_prueba_par:
            test_img = pad_image(images[idx], final_size)
            test_img_norm = zero_mean_normalize(test_img)
            true_label = int(labels[idx])

            int_A = np.max(optical_correlator(test_img_norm, reference_images_sdf[clase_A]))
            int_B = np.max(optical_correlator(test_img_norm, reference_images_sdf[clase_B]))

            predicted_digit = clase_A if int_A > int_B else clase_B
            y_true.append(true_label)
            y_pred.append(predicted_digit)

        # Calculamos y guardamos el Accuracy en la matriz
        acc = accuracy_score(y_true, y_pred)
        accuracy_matrix[clase_A, clase_B] = round(acc, 2)
        print(f"     Accuracy final ({clase_A} vs {clase_B}): {acc:.2f}")

elapsed_time_total = time.time() - start_time_total
print(f"\n==========================================")
print(f"TIEMPO TOTAL DEL BUCLE: {elapsed_time_total / 60:.2f} minutos")
print(f"==========================================\n")

columnas = [str(i) for i in range(10)]
df_accuracy = pd.DataFrame(accuracy_matrix, columns=columnas, index=columnas)

print("MATRIZ DE ACCURACY DE CLASIFICACIÓN BINARIA SDF:\n")
print(df_accuracy.to_string())

Iniciando separación estructurada de datos (Clases 0 al 9)...

Iniciando síntesis iterativa y evaluación de matriz SDF...

---> Evaluando par: 0 vs 1
     Accuracy final (0 vs 1): 0.99

---> Evaluando par: 0 vs 2
     Accuracy final (0 vs 2): 0.80

---> Evaluando par: 0 vs 3
     Accuracy final (0 vs 3): 0.84

---> Evaluando par: 0 vs 4
     Accuracy final (0 vs 4): 0.94

---> Evaluando par: 0 vs 5
     Accuracy final (0 vs 5): 0.82

---> Evaluando par: 0 vs 6
     Accuracy final (0 vs 6): 0.85

---> Evaluando par: 0 vs 7
     Accuracy final (0 vs 7): 0.89

---> Evaluando par: 0 vs 8
     Accuracy final (0 vs 8): 0.95

---> Evaluando par: 0 vs 9
     Accuracy final (0 vs 9): 0.96

---> Evaluando par: 1 vs 2
     Accuracy final (1 vs 2): 0.85

---> Evaluando par: 1 vs 3
     Accuracy final (1 vs 3): 0.82

---> Evaluando par: 1 vs 4
     Accuracy final (1 vs 4): 0.96

---> Evaluando par: 1 vs 5
     Accuracy final (1 vs 5): 0.94

---> Evaluando par: 1 vs 6
     Accuracy final (1 vs 6): 0